<a href="https://colab.research.google.com/github/ArwaFadaaq/Robust-Multimodal-Age-Verification-for-Online-Gaming-Chat/blob/main/GP2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install silero-vad

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 88.4 MB/s eta 0:00:00


In [ ]:
import kagglehub
import os
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchaudio
import librosa
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from preprocessing import preprocess_audio, save_segment_as_wav

ModuleNotFoundError: No module named 'preprocessing'

https://www.kaggle.com/datasets/mozillaorg/common-voice/data

In [ ]:
# ===== Dataset 1: Common Voice =====

# Download the dataset from Kaggle
COMMON_VOICE_NAME = "mozillaorg/common-voice"
path = kagglehub.dataset_download(COMMON_VOICE_NAME)

# Show dataset path
print("Path:", path)

# Load the metadata CSV file
csv_path = os.path.join(path, "cv-valid-train.csv")
df = pd.read_csv(csv_path)

# List files in the dataset folder
print(os.listdir(path))

# Show first 3 rows of the dataset
df.head(3)

In [ ]:
# Function to display age and gender distribution in the dataset
def show_metadata_distribution(df: pd.DataFrame, title: str):

    # Print section title
    print(f"\n=== {title} ===")

    # Show age distribution if age column exists
    if 'age' in df.columns:
        age_counts = df['age'].value_counts(dropna=False)
        print("\nAge distribution:")
        print(age_counts)

    # Show gender distribution if gender column exists
    if 'gender' in df.columns:
        gender_counts = df['gender'].value_counts(dropna=False)
        print("\nGender distribution:")
        print(gender_counts)

    # Print total number of rows
    print("\nNumber of rows:", len(df))


# Display dataset statistics before cleaning
show_metadata_distribution(df, "BEFORE CLEANING")

In [ ]:
# Function to clean Common Voice metadata
def clean_common_voice_metadata(df: pd.DataFrame) -> pd.DataFrame:

    # Remove rows with missing age or gender
    df_clean = df.dropna(subset=['age', 'gender']).copy()

    # Remove samples where gender is labeled as 'other'
    df_clean = df_clean[df_clean['gender'] != 'other']

    # Convert age labels into two classes: child vs adult
    df_clean['age'] = df_clean['age'].apply(
        lambda x: 'child' if x == 'teens' else 'adult'
    )

    return df_clean

# Apply cleaning
df_clean = clean_common_voice_metadata(df)

# Show distribution after cleaning
show_metadata_distribution(df_clean, "AFTER CLEANING")

In [ ]:
df_clean['audio_path'] = df_clean['filename'].apply(
    lambda x: os.path.join(path, "cv-valid-train", x)
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
processed_dir = "/content/drive/MyDrive/age_verification/data/processed/clean"
os.makedirs(processed_dir, exist_ok=True)

segment_records = []

dataset_name = "cv"
spoof_label = 0   # 0 = clean


def gender_to_code(gender: str) -> str:
    if gender == "male":
        return "m"
    elif gender == "female":
        return "f"
    return "u"


for idx, row in tqdm(df_clean.iterrows(), total=len(df_clean), desc="Processing Common Voice"):

    audio_path = row["audio_path"]
    age_label = row["age"]
    gender_label = row["gender"]

    gender_code = gender_to_code(gender_label)

    try:
        segments = preprocess_audio(
            audio_path=audio_path,
            remove_internal_silence=False
        )

        if len(segments) == 0:
            continue

        for seg_idx in range(len(segments)):

            file_name = f"{dataset_name}_{gender_code}_{spoof_label}_{idx}_seg_{seg_idx}.wav"

            segment_path = os.path.join(processed_dir, file_name)

            save_segment_as_wav(
                segment=segments[seg_idx],
                output_path=segment_path
            )

            segment_records.append({
                "original_audio_path": audio_path,
                "segment_path": segment_path,
                "dataset": dataset_name,
                "age": age_label,
                "gender": gender_label,
                "spoof_label": spoof_label,
                "sample_id": idx,
                "segment_id": seg_idx
            })

    except Exception as e:
        print(f"Error processing {audio_path}: {e}")


segments_df = pd.DataFrame(segment_records)

print("Total saved segments:", len(segments_df))
segments_df.head()

Processing Common Voice: 100%|██████████| 72692/72692 [25:08<00:00, 48.19it/s]


Total saved segments: 50418


,original_audio_path,segment_path,dataset,age,gender,spoof_label,sample_id,segment_id
0,/kaggle/input/common-voice/cv-valid-train/cv-v...,/content/drive/MyDrive/age_verification/data/p...,cv,adult,female,0,5,0
1,/kaggle/input/common-voice/cv-valid-train/cv-v...,/content/drive/MyDrive/age_verification/data/p...,cv,adult,female,0,13,0
2,/kaggle/input/common-voice/cv-valid-train/cv-v...,/content/drive/MyDrive/age_verification/data/p...,cv,adult,male,0,14,0
3,/kaggle/input/common-voice/cv-valid-train/cv-v...,/content/drive/MyDrive/age_verification/data/p...,cv,adult,male,0,14,1
4,/kaggle/input/common-voice/cv-valid-train/cv-v...,/content/drive/MyDrive/age_verification/data/p...,cv,adult,male,0,19,0


In [ ]:
metadata_dir = "/content/drive/MyDrive/age_verification/data/metadata"
os.makedirs(metadata_dir, exist_ok=True)

metadata_path = os.path.join(metadata_dir, "common_voice_clean_segments.csv")

segments_df.to_csv(metadata_path, index=False)

print("Metadata saved to:", metadata_path)

Metadata saved to: /content/drive/MyDrive/age_verification/data/metadata/common_voice_clean_segments.csv
